In [1]:
try:
    import transformers
    import datasets
    from transformers import BitsAndBytesConfig
    from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
    from peft import TaskType, PeftConfig, PeftModel 
except:
    print("all or at least one of the package not installed. Installing now. Please remember to restart kernel once done")
    !pip install transformers bitsandbytes accelerate peft
    !pip install datasets==3.3.2

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

In [3]:
import torch
from torch.utils.data import DataLoader, Dataset

from functools import partial

# TODO: Import any packages that you might need
#importing the "King" library :P
import sagemaker
import boto3
from sagemaker.inputs import TrainingInput

#sagemaker pytorch container estimator related libraries
from sagemaker.pytorch import PyTorch
from sagemaker.pytorch import PyTorchModel


from sagemaker.estimator import Estimator
from sagemaker.model import Model
from sagemaker.predictor import Predictor

#general utility imports
import os
import glob
import pandas as pd
import numpy as np
import random

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [4]:
ds = load_dataset("ingeniumacademy/reuters_articles")

README.md:   0%|          | 0.00/698 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/8.13M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/827k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/17262 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2158 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2158 [00:00<?, ? examples/s]

In [5]:
ds

DatasetDict({
    train: Dataset({
        features: ['title', 'body'],
        num_rows: 17262
    })
    validation: Dataset({
        features: ['title', 'body'],
        num_rows: 2158
    })
    test: Dataset({
        features: ['title', 'body'],
        num_rows: 2158
    })
})

In [6]:
def create_full_article_col(row):
    full_article = f'TITLE: {row["title"]}\n\nBODY: {row["body"]}'
    return {'full_article': full_article}

In [7]:
ds_1 = ds.map(create_full_article_col)
ds_1

Map:   0%|          | 0/17262 [00:00<?, ? examples/s]

Map:   0%|          | 0/2158 [00:00<?, ? examples/s]

Map:   0%|          | 0/2158 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['title', 'body', 'full_article'],
        num_rows: 17262
    })
    validation: Dataset({
        features: ['title', 'body', 'full_article'],
        num_rows: 2158
    })
    test: Dataset({
        features: ['title', 'body', 'full_article'],
        num_rows: 2158
    })
})

In [8]:
print(ds_1['train'][1]['full_article'])

TITLE: STANDARD OIL <SRD> TO FORM FINANCIAL UNIT

BODY: Standard Oil Co and BP North America
Inc said they plan to form a venture to manage the money market
borrowing and investment activities of both companies.
    BP North America is a subsidiary of British Petroleum Co
Plc <BP>, which also owns a 55 pct interest in Standard Oil.
    The venture will be called BP/Standard Financial Trading
and will be operated by Standard Oil under the oversight of a
joint management committee.

 Reuter



In [9]:
print(ds_1['train'][3]['full_article'])

TITLE: TALKING POINT/BANKAMERICA <BAC> EQUITY OFFER

BODY: BankAmerica Corp is not under
pressure to act quickly on its proposed equity offering and
would do well to delay it because of the stock's recent poor
performance, banking analysts said.
    Some analysts said they have recommended BankAmerica delay
its up to one-billion-dlr equity offering, which has yet to be
approved by the Securities and Exchange Commission.
    BankAmerica stock fell this week, along with other banking
issues, on the news that Brazil has suspended interest payments
on a large portion of its foreign debt.
    The stock traded around 12, down 1/8, this afternoon,
after falling to 11-1/2 earlier this week on the news.
    Banking analysts said that with the immediate threat of the
First Interstate Bancorp <I> takeover bid gone, BankAmerica is
under no pressure to sell the securities into a market that
will be nervous on bank stocks in the near term.
    BankAmerica filed the offer on January 26. It was seen a

In [10]:
ckpt = 'gpt2-medium'
tokenizer = AutoTokenizer.from_pretrained(ckpt)
tokenizer

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

GPT2Tokenizer(name_or_path='gpt2-medium', vocab_size=50257, model_max_length=1024, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
})

In [11]:
tokenizer.pad_token, tokenizer.eos_token

(None, '<|endoftext|>')

In [12]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token, tokenizer.eos_token

('<|endoftext|>', '<|endoftext|>')

In [13]:
tokenizer.encode('<|endoftext|>')

[50256]

In [14]:
!pip install torchinfo

In [15]:
from torchinfo import summary

In [16]:
config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

In [17]:
quantization_config = BitsAndBytesConfig(
                load_in_8bit=True,
                )

In [18]:
gpt2_med_model_4bit = AutoModelForCausalLM.from_pretrained(ckpt, quantization_config=config)
gp2_med_model_8bit = AutoModelForCausalLM.from_pretrained(ckpt, quantization_config=quantization_config)

#this ran successfully in gpu instance so should be ok

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

In [19]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [20]:
def net(args, ckpt, quant_bits, device):
    config = None
    lora_config = LoraConfig(
        lora_alpha=32,
        lora_dropout=0.1,
        r=16,
        task_type="CAUSAL_LM" # TaskType.CAUSAL_LM",
        )
    config_4bit = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16
        )
    config_8bit = BitsAndBytesConfig(
        load_in_8bit=True,
        )
    orig_model = AutoModelForCausalLM.from_pretrained(ckpt)
    orig_model.config.pad_token_id = orig_model.config.eos_token_id
    if args == "lora":
        if quant_bits == 4:
            config = config_4bit
        elif quant_bits == 8:
            config = config_8bit
        else:
            config = None

    if config:
        print("Model Quantization requested")
        quant_model = AutoModelForCausalLM.from_pretrained(
            ckpt, 
            quantization_config=config
            )
        quant_model.config.pad_token_id = quant_model.config.eos_token_id
        quant_model.gradient_checkpointing_enable()
        quant_model = prepare_model_for_kbit_training(quant_model)
        model = get_peft_model(quant_model, lora_config)
    elif args == "lora":
        print("Only PEFT. No Model Quantization requested")
        model = get_peft_model(orig_model, lora_config)
    else:
        print("NO PEFT. No Model Quantization requested")
        model = orig_model

    s = summary(model)
    print(f"Total number of trainable parameters: {s.__dict__['trainable_params']}")
    model.to(device)
    return model

s=summary(model)
print(s.__dict__['total_params']) #: 406286336,
print(s.__dict__['trainable_params']) #: 406286336,
print(s.__dict__['total_param_bytes']) #: 1625145344,

s.__dic

In [21]:
args = "lora"
ckpt = "gpt2-medium"
quant_bits = 4

In [22]:
model = net(args, ckpt, quant_bits, device)

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Model Quantization requested


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Total number of trainable parameters: 1572864


In [23]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GPT2LMHeadModel(
      (transformer): GPT2Model(
        (wte): Embedding(50257, 1024)
        (wpe): Embedding(1024, 1024)
        (drop): Dropout(p=0.1, inplace=False)
        (h): ModuleList(
          (0-23): 24 x GPT2Block(
            (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (attn): GPT2Attention(
              (c_attn): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1024, out_features=3072, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (l

In [24]:
args = "lora"
ckpt = "gpt2-medium"
quant_bits = 8

In [25]:
del model
model = net(args, ckpt, quant_bits, device)

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Model Quantization requested


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Total number of trainable parameters: 1572864


In [26]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GPT2LMHeadModel(
      (transformer): GPT2Model(
        (wte): Embedding(50257, 1024)
        (wpe): Embedding(1024, 1024)
        (drop): Dropout(p=0.1, inplace=False)
        (h): ModuleList(
          (0-23): 24 x GPT2Block(
            (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (attn): GPT2Attention(
              (c_attn): lora.Linear8bitLt(
                (base_layer): Linear8bitLt(in_features=1024, out_features=3072, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
              

In [27]:
args = "lora"
ckpt = "gpt2-medium"
quant_bits = 0

In [28]:
del model
model = net(args, ckpt, quant_bits, device)

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Only PEFT. No Model Quantization requested
Total number of trainable parameters: 1572864


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [29]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GPT2LMHeadModel(
      (transformer): GPT2Model(
        (wte): Embedding(50257, 1024)
        (wpe): Embedding(1024, 1024)
        (drop): Dropout(p=0.1, inplace=False)
        (h): ModuleList(
          (0-23): 24 x GPT2Block(
            (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (attn): GPT2Attention(
              (c_attn): lora.Linear(
                (base_layer): Conv1D(nf=3072, nx=1024)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
    

In [30]:
args = None
ckpt = "gpt2-medium"
quant_bits = 0

In [31]:
del model
model = net(args, ckpt, quant_bits, device)

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

NO PEFT. No Model Quantization requested
Total number of trainable parameters: 406286336


In [32]:
args = "lora"
ckpt = "gpt2"
quant_bits = 4

In [33]:
del model
model = net(args, ckpt, quant_bits, device)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model Quantization requested


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Total number of trainable parameters: 589824


In [34]:
args = "lora"
ckpt = "gpt2"
quant_bits = 8

In [35]:
del model
model = net(args, ckpt, quant_bits, device)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model Quantization requested


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Total number of trainable parameters: 589824


In [36]:
args = "lora"
ckpt = "gpt2"
quant_bits = 0

In [37]:
del model
model = net(args, ckpt, quant_bits, device)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Only PEFT. No Model Quantization requested
Total number of trainable parameters: 589824


In [38]:
args = None
ckpt = "gpt2"
quant_bits = 0

In [39]:
del model
model = net(args, ckpt, quant_bits, device)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

NO PEFT. No Model Quantization requested
Total number of trainable parameters: 163037184


In [40]:
df = ds_1['train'].to_pandas()

In [41]:
df['full_article'].describe()

count                 17262
unique                16338
top       TITLE: \n\nBODY: 
freq                    718
Name: full_article, dtype: object

In [42]:
len(df)

17262

In [43]:
df

,title,body,full_article
0,BAHIA COCOA REVIEW,Showers continued throughout the week in\nthe ...,TITLE: BAHIA COCOA REVIEW\n\nBODY: Showers con...
1,STANDARD OIL <SRD> TO FORM FINANCIAL UNIT,Standard Oil Co and BP North America\nInc said...,TITLE: STANDARD OIL <SRD> TO FORM FINANCIAL UN...
2,TEXAS COMMERCE BANCSHARES <TCB> FILES PLAN,Texas Commerce Bancshares Inc's Texas\nCommerc...,TITLE: TEXAS COMMERCE BANCSHARES <TCB> FILES P...
3,TALKING POINT/BANKAMERICA <BAC> EQUITY OFFER,BankAmerica Corp is not under\npressure to act...,TITLE: TALKING POINT/BANKAMERICA <BAC> EQUITY ...
4,NATIONAL AVERAGE PRICES FOR FARMER-OWNED RESERVE,The U.S. Agriculture Department\nreported the ...,TITLE: NATIONAL AVERAGE PRICES FOR FARMER-OWNE...
...,...,...,...
17257,BANK OF JAPAN TO SELL 600 BILLION YEN IN BILLS,The Bank of Japan will sell 600 billion\nyen i...,TITLE: BANK OF JAPAN TO SELL 600 BILLION YEN I...
17258,"FURTHER TERM FOR POEHL LIKELY, BONN SOURCES SAY",Karl Otto Poehl is likely to be re-elected\nPr...,"TITLE: FURTHER TERM FOR POEHL LIKELY, BONN SOU..."
17259,TAIWAN ISSUES MORE CDS TO CURB MONEY SUPPLY GR...,The central bank issued 7.53 billion\nTaiwan d...,TITLE: TAIWAN ISSUES MORE CDS TO CURB MONEY SU...
17260,JAPAN HOLDS OUT PROMISE OF FUNDS FOR ASIAN BANK,Japanese Finance Minister Kiichi\nMiyazawa ope...,TITLE: JAPAN HOLDS OUT PROMISE OF FUNDS FOR AS...


In [44]:
df.columns

Index(['title', 'body', 'full_article'], dtype='object')

In [45]:
tokenizer.encode(tokenizer.eos_token)

[50256]

In [46]:
class dset(Dataset):
    def __init__(self, df, tokenizer, max_length):
        super().__init__()
        self.data = df['full_article']
        self.tokenizer = tokenizer
        self.eos_tokid = tokenizer.encode(tokenizer.eos_token)
        self.max_length = max_length

    
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data.iloc[idx]
        x = self.tokenizer(x, max_length=self.max_length, truncation=True)
        y = x['input_ids'][1:] + self.eos_tokid
        return x, y

In [47]:
ds = dset(df, tokenizer, 512)

In [48]:
len(ds)

17262

In [49]:
ds[0][0]

{'input_ids': [49560, 2538, 25, 347, 18429, 3539, 327, 4503, 23621, 4526, 28206, 198, 198, 33, 33076, 25, 911, 3618, 3767, 3690, 262, 1285, 287, 198, 1169, 13081, 544, 35845, 6516, 11, 23863, 26336, 262, 18393, 1201, 1903, 198, 21339, 290, 10068, 13285, 329, 262, 2406, 10042, 5488, 11, 198, 16670, 3487, 27716, 2974, 423, 407, 587, 15032, 11, 198, 5377, 747, 10312, 4176, 531, 287, 663, 10273, 2423, 13, 198, 220, 220, 220, 383, 5894, 2278, 1724, 262, 10042, 5488, 481, 307, 2739, 428, 614, 13, 198, 220, 220, 220, 27350, 12786, 329, 262, 1285, 4444, 3945, 2534, 547, 20708, 11, 26115, 11668, 198, 1659, 3126, 8769, 418, 1642, 257, 23818, 2472, 329, 262, 1622, 286, 642, 13, 6052, 198, 4029, 77, 1028, 642, 13, 6659, 379, 262, 976, 3800, 938, 614, 13, 6521, 340, 2331, 198, 5562, 35845, 6793, 2961, 319, 762, 16747, 373, 3017, 287, 262, 198, 283, 380, 12786, 5538, 13, 198, 220, 220, 220, 955, 747, 10312, 4176, 531, 612, 318, 991, 617, 4719, 355, 284, 703, 198, 29482, 1468, 13833, 35845, 318, 991,

In [50]:
ds[0][1]

[2538,
 25,
 347,
 18429,
 3539,
 327,
 4503,
 23621,
 4526,
 28206,
 198,
 198,
 33,
 33076,
 25,
 911,
 3618,
 3767,
 3690,
 262,
 1285,
 287,
 198,
 1169,
 13081,
 544,
 35845,
 6516,
 11,
 23863,
 26336,
 262,
 18393,
 1201,
 1903,
 198,
 21339,
 290,
 10068,
 13285,
 329,
 262,
 2406,
 10042,
 5488,
 11,
 198,
 16670,
 3487,
 27716,
 2974,
 423,
 407,
 587,
 15032,
 11,
 198,
 5377,
 747,
 10312,
 4176,
 531,
 287,
 663,
 10273,
 2423,
 13,
 198,
 220,
 220,
 220,
 383,
 5894,
 2278,
 1724,
 262,
 10042,
 5488,
 481,
 307,
 2739,
 428,
 614,
 13,
 198,
 220,
 220,
 220,
 27350,
 12786,
 329,
 262,
 1285,
 4444,
 3945,
 2534,
 547,
 20708,
 11,
 26115,
 11668,
 198,
 1659,
 3126,
 8769,
 418,
 1642,
 257,
 23818,
 2472,
 329,
 262,
 1622,
 286,
 642,
 13,
 6052,
 198,
 4029,
 77,
 1028,
 642,
 13,
 6659,
 379,
 262,
 976,
 3800,
 938,
 614,
 13,
 6521,
 340,
 2331,
 198,
 5562,
 35845,
 6793,
 2961,
 319,
 762,
 16747,
 373,
 3017,
 287,
 262,
 198,
 283,
 380,
 12786,
 5538,
 13,


In [51]:
ds[1][0]['input_ids'][1:] == ds[1][1][:-1]

True

In [52]:
type(ds[1][0]['input_ids'])

list

In [53]:
type(ds[1][1])

list

In [54]:
len(ds[0])

2

In [55]:
def collate_fn(batch, device, tokenizer):
    pad_tokenid = tokenizer.encode(tokenizer.eos_token)
    inp_list, att_list, y_list = [],  [], []
    print(batch)
    len_labs =  [len(l) for i, l in batch]
    print(len_labs)
    max_len = max(len_labs)
    print(f'Max Length is {max_len}')
    for _text, _label in batch:
        inp_ids = _text['input_ids']
        am = _text['attention_mask']
        label = _label
        if len(inp_ids) < max_len:
            deficit = max_len - len(inp_ids)
            inp_ids = inp_ids + pad_tokenid*deficit
            am = am + [0]*deficit
            label = label + [-100]*deficit
        inp_list.append(torch.tensor(inp_ids))
        att_list.append(torch.tensor(am))
        y_list.append(torch.tensor(label))
    input_ids = torch.vstack(inp_list)
    input_ids = input_ids.to(device)
    attention_mask = torch.vstack(att_list)
    attention_mask = attention_mask.to(device)
    labels = torch.vstack(y_list)
    labels = labels.to(device)
    return {'input_ids': input_ids, 'attention_mask': attention_mask}, labels

In [56]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [57]:
wrapper_collate_fn = partial(
    collate_fn,
    device=device,
    tokenizer=tokenizer
)

In [58]:
tokenizer

GPT2Tokenizer(name_or_path='gpt2-medium', vocab_size=50257, model_max_length=1024, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
})

In [59]:
dl = DataLoader(ds, shuffle=False, batch_size=2, collate_fn=wrapper_collate_fn)

In [60]:
iter_dl = iter(dl)

In [61]:
i,l = next(iter_dl)

[({'input_ids': [49560, 2538, 25, 347, 18429, 3539, 327, 4503, 23621, 4526, 28206, 198, 198, 33, 33076, 25, 911, 3618, 3767, 3690, 262, 1285, 287, 198, 1169, 13081, 544, 35845, 6516, 11, 23863, 26336, 262, 18393, 1201, 1903, 198, 21339, 290, 10068, 13285, 329, 262, 2406, 10042, 5488, 11, 198, 16670, 3487, 27716, 2974, 423, 407, 587, 15032, 11, 198, 5377, 747, 10312, 4176, 531, 287, 663, 10273, 2423, 13, 198, 220, 220, 220, 383, 5894, 2278, 1724, 262, 10042, 5488, 481, 307, 2739, 428, 614, 13, 198, 220, 220, 220, 27350, 12786, 329, 262, 1285, 4444, 3945, 2534, 547, 20708, 11, 26115, 11668, 198, 1659, 3126, 8769, 418, 1642, 257, 23818, 2472, 329, 262, 1622, 286, 642, 13, 6052, 198, 4029, 77, 1028, 642, 13, 6659, 379, 262, 976, 3800, 938, 614, 13, 6521, 340, 2331, 198, 5562, 35845, 6793, 2961, 319, 762, 16747, 373, 3017, 287, 262, 198, 283, 380, 12786, 5538, 13, 198, 220, 220, 220, 955, 747, 10312, 4176, 531, 612, 318, 991, 617, 4719, 355, 284, 703, 198, 29482, 1468, 13833, 35845, 318, 99

In [62]:
i['input_ids'].shape, i['attention_mask'].shape, l.shape

(torch.Size([2, 512]), torch.Size([2, 512]), torch.Size([2, 512]))

In [63]:
i['input_ids']

tensor([[49560,  2538,    25,  ...,    11,  7029,   290],
        [49560,  2538,    25,  ..., 50256, 50256, 50256]], device='cuda:0')

In [64]:
i['attention_mask']

tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0]], device='cuda:0')

In [65]:
i['input_ids'] = i['input_ids'].to(device)
i['attention_mask'] = i['attention_mask'].to(device)

In [66]:
out = model(**i)

In [67]:
out.logits.shape, out

(torch.Size([2, 512, 50257]),
 CausalLMOutputWithCrossAttentions(loss=None, logits=tensor([[[ -32.5351,  -31.3880,  -33.9832,  ...,  -42.8560,  -42.2166,
            -33.0416],
          [ -91.4329,  -90.0395,  -91.3562,  ..., -102.3688, -100.5219,
            -92.0736],
          [ -69.2479,  -68.6931,  -68.9566,  ...,  -76.6611,  -75.7209,
            -66.8522],
          ...,
          [-101.0089,  -99.4432,  -99.8513,  ..., -107.8896, -108.4641,
            -97.9067],
          [ -50.4815,  -52.6545,  -53.9596,  ...,  -59.7929,  -60.0427,
            -49.6024],
          [ -77.5360,  -78.3590,  -79.1278,  ...,  -85.0720,  -82.5483,
            -74.6692]],
 
         [[ -32.5351,  -31.3880,  -33.9832,  ...,  -42.8560,  -42.2166,
            -33.0416],
          [ -91.4329,  -90.0395,  -91.3562,  ..., -102.3688, -100.5219,
            -92.0736],
          [ -69.2479,  -68.6931,  -68.9566,  ...,  -76.6611,  -75.7209,
            -66.8522],
          ...,
          [ -95.4947,  -88.235

In [68]:
tokenizer.decode([49560,  2538,    25])

'TITLE:'

In [69]:
tokenizer.decode([797, 11894,   198,   191])

' Reuter\n\x03'

In [70]:
#df.to_csv('./reuters_ds.csv', index=False)

In [71]:
vocab_size =  tokenizer.vocab_size
vocab_size

50257

#### TILL THIS POINT EXECUTE IN GPU NOTEBOOK REST EXECUTE IN T3.MEDIUM. RUN INFERENCES IN T3.XLARGE

Strategy for training experiments:
1) We have already executed full fine tuning of gpt2 model for 4 epochs
2) Next, we do 4 bit fine-tuning for gpt2 for 4 epochs and measure run times. if lesser then execute for more epochs. Max time limit : 1 hour --> 2 experiments: 1-2 hours
3) Next we do 8 bit fine-tuning for 4 epochs. Same procedure as above 1 hour
4) Next we do 4 bit for gpt2-medium followed by 8 bit. 2-3 hours
5) Next we run inference and check the quality of the outputs produced by different models- total 4 experiments

Next we do QA and summarization fine tuning from another udemy course --> instruction fine tuning
We can use karthick dataset from AWS LLM course as well

In [10]:
#Sagemaker related commands
sess = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sess.boto_region_name
bucket = sess.default_bucket()
sess, role, region, bucket

(<sagemaker.session.Session at 0x7f8c8904fc10>,
 'arn:aws:iam::191013407134:role/service-role/AmazonSageMaker-ExecutionRole-20250124T222384',
 'us-east-1',
 'sagemaker-us-east-1-191013407134')

In [11]:
%pwd

'/home/ec2-user/SageMaker/CURRENTFOCUS/LLMS_transformers_WIP/LLMfromScratch/NLP_tasks/FineTuneDecoder'

In [12]:
S3_LOC = f's3://{bucket}/gpt_train/dataset/reuters/'
S3_LOC

's3://sagemaker-us-east-1-191013407134/gpt_train/dataset/reuters/'

In [13]:
#!aws s3 cp ./reuters_ds.csv {S3_LOC}

In [14]:
!aws s3 ls --recursive {S3_LOC}

2025-02-26 05:00:50   27728622 gpt_train/dataset/reuters/reuters_ds.csv


In [15]:
# let us get image uri
training_image_uri = sagemaker.image_uris.retrieve(
    framework='pytorch', 
    version='2.0',
    instance_type='ml.g4dn.xlarge',
    region=region,
    py_version='py310',
    image_scope='training'
)
print(training_image_uri)

763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-training:2.0-gpu-py310


In [16]:
train_uri = f'{S3_LOC}'
train_uri

's3://sagemaker-us-east-1-191013407134/gpt_train/dataset/reuters/'

In [17]:
s3_inp_tr = TrainingInput(
    s3_data = train_uri
    )
s3_inp_tr

In [18]:
data_channels = {
    'train': s3_inp_tr
    }
data_channels

{'train': <sagemaker.inputs.TrainingInput at 0x7f8c62049270>}

In [19]:
objective_metric_name = "average training loss"
objective_type = "Minimize"
metric_definitions = [{"Name": "average training loss", "Regex": "Average loss: ([0-9\\.]+)"},
                      {"Name": "Perplexity", "Regex": "Perplexity: ([0-9\\.]+)"}]
objective_metric_name, objective_type, metric_definitions

('average training loss',
 'Minimize',
 [{'Name': 'average training loss', 'Regex': 'Average loss: ([0-9\\.]+)'},
  {'Name': 'Perplexity', 'Regex': 'Perplexity: ([0-9\\.]+)'}])

In [20]:
ic=1
i_type = "ml.g4dn.xlarge"
ic, i_type

(1, 'ml.g4dn.xlarge')

In [21]:
%pwd

'/home/ec2-user/SageMaker/CURRENTFOCUS/LLMS_transformers_WIP/LLMfromScratch/NLP_tasks/FineTuneDecoder'

In [22]:
hparams = {
    'bs': 16,
    'lrate': 0.0006,
    'num_epochs': 4,
    'environ': 'aws',
    'block_size': 256,
    'ckpt': 'gpt2',
    'quant_bits': 4,
    'peft': 'lora'
}
hparams

{'bs': 16,
 'lrate': 0.0006,
 'num_epochs': 4,
 'environ': 'aws',
 'block_size': 256,
 'ckpt': 'gpt2',
 'quant_bits': 4,
 'peft': 'lora'}

In [23]:
est = Estimator(
    image_uri=training_image_uri,
    role=role,
    source_dir='./scripts',
    entry_point='train-gpt2-reuters.py',
    instance_count=ic,
    instance_type=i_type,
    base_job_name='reutersgpt-train',
    hyperparameters=hparams,
    metric_definitions=metric_definitions,
    objective_type=objective_type,
    objective_metric_name=objective_metric_name
)

est.fit(
    inputs=data_channels,
    wait=True
)

INFO:sagemaker:Creating training-job with name: reutersgpt-train-2026-04-26-03-52-17-073


2026-04-26 03:52:17 Starting - Starting the training job...
2026-04-26 03:52:43 Starting - Preparing the instances for training...
2026-04-26 03:53:07 Downloading - Downloading input data...
2026-04-26 03:53:38 Downloading - Downloading the training image..................
2026-04-26 03:56:50 Training - Training image download completed. Training in progress...bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
2026-04-26 03:57:01,940 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2026-04-26 03:57:01,958 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-04-26 03:57:01,973 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succeed.
2026-04-26 03:57:01,982 sagemaker_pytorch_container.training INFO     Invoking user training script.
2026-04-26 03:57:03,948 sagemaker-training-toolkit INFO     Installing dependencies

In [24]:
est.latest_training_job.describe()['FinalMetricDataList']

[{'MetricName': 'Perplexity',
  'Value': 10.314106941223145,
  'Timestamp': datetime.datetime(2026, 4, 26, 4, 50, 59, tzinfo=tzlocal())},
 {'MetricName': 'average training loss',
  'Value': 2.333512544631958,
  'Timestamp': datetime.datetime(2026, 4, 26, 4, 50, 59, tzinfo=tzlocal())}]

In [25]:
!aws s3 ls --recursive {est.model_data}

2026-04-26 04:51:17    3425537 reutersgpt-train-2026-04-26-03-52-17-073/output/model.tar.gz


In [26]:
!aws s3 cp {est.model_data} ./temp/gpt2-lora-4-bit-quant.tar.gz

download: s3://sagemaker-us-east-1-191013407134/reutersgpt-train-2026-04-26-03-52-17-073/output/model.tar.gz to temp/gpt2-lora-4-bit-quant.tar.gz


In [27]:
%cd temp

/home/ec2-user/SageMaker/CURRENTFOCUS/LLMS_transformers_WIP/LLMfromScratch/NLP_tasks/FineTuneDecoder/temp


In [28]:
!ls -ltrh

total 4.9M
-rw-rw-r-- 1 ec2-user ec2-user 756K Feb 28  2025 Quant_ReuterSDSgpt2fine-tuning (2).ipynb
-rw-rw-r-- 1 ec2-user ec2-user 840K Feb 28  2025 Quant_ReuterSDSgpt2fine-tuning (1).ipynb
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  1  2025 ckpt_Epoch_4_Prplxty_1.0384616768394737
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  1  2025 ckpt_Epoch_4_Prplxty_1.0191842654468437
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  1  2025 ckpt_Epoch_4_Prplxty_1.00977148872375
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  1  2025 ckpt_Epoch_2_Prplxty_9.256776574458915
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  2  2025 ckpt_Epoch_2_Prplxty_8.363786833972586
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  2  2025 ckpt_Epoch_2_Prplxty_8.29573556990836
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  4  2025 ckpt_Epoch_20_Prplxty_4.906924388591055
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  4  2025 ckpt_Epoch_30_Prplxty_3.6311811785448933
drwxrwxr-x 3 ec2-user ec2-user 4.0K Mar  5  2025 train
-rw-rw-r-- 1 ec2-user ec2-user 3.3M Apr 26 04:5

In [29]:
#!gunzip -dc model.tar.gz |tar xvf -

In [30]:
#!ls -ltrh

##### We have some observations:
- Training run time was lower than full fine tuning --> can be attributed to gradient accumulation
- Differential model is very small (less than 4 MB)

### Experiment 2: Run GPT2 with 8 bits. We will try to use 8 steps in gradient accumulation and slightly higher learning rate

In [31]:
%cd ..
%pwd

/home/ec2-user/SageMaker/CURRENTFOCUS/LLMS_transformers_WIP/LLMfromScratch/NLP_tasks/FineTuneDecoder


'/home/ec2-user/SageMaker/CURRENTFOCUS/LLMS_transformers_WIP/LLMfromScratch/NLP_tasks/FineTuneDecoder'

In [32]:
hparams = {
    'bs': 16,
    'lrate': 0.0008,
    'num_epochs': 4,
    'environ': 'aws',
    'block_size': 256,
    'ckpt': 'gpt2',
    'quant_bits': 8,
    'peft': 'lora',
    'grad_accum_steps': 8
}
hparams

{'bs': 16,
 'lrate': 0.0008,
 'num_epochs': 4,
 'environ': 'aws',
 'block_size': 256,
 'ckpt': 'gpt2',
 'quant_bits': 8,
 'peft': 'lora',
 'grad_accum_steps': 8}

In [33]:
est_gpt2_8bit = Estimator(
    image_uri=training_image_uri,
    role=role,
    source_dir='./scripts',
    entry_point='train-gpt2-reuters.py',
    instance_count=ic,
    instance_type=i_type,
    base_job_name='reutersgpt-train',
    hyperparameters=hparams,
    metric_definitions=metric_definitions,
    objective_type=objective_type,
    objective_metric_name=objective_metric_name
)

est_gpt2_8bit.fit(
    inputs=data_channels,
    wait=True
)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: reutersgpt-train-2026-04-26-05-01-11-367


2026-04-26 05:01:11 Starting - Starting the training job...
2026-04-26 05:01:35 Starting - Preparing the instances for training...
2026-04-26 05:02:02 Downloading - Downloading input data...
2026-04-26 05:02:33 Downloading - Downloading the training image....................
2026-04-26 05:05:50 Training - Training image download completed. Training in progress.bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
2026-04-26 05:05:59,536 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2026-04-26 05:05:59,557 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-04-26 05:05:59,567 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succeed.
2026-04-26 05:05:59,574 sagemaker_pytorch_container.training INFO     Invoking user training script.
2026-04-26 05:06:01,576 sagemaker-training-toolkit INFO     Installing dependencies

In [34]:
est_gpt2_8bit.latest_training_job.describe()['FinalMetricDataList']

[{'MetricName': 'Perplexity',
  'Value': 11.32615852355957,
  'Timestamp': datetime.datetime(2026, 4, 26, 5, 57, 9, tzinfo=tzlocal())},
 {'MetricName': 'average training loss',
  'Value': 2.427114963531494,
  'Timestamp': datetime.datetime(2026, 4, 26, 5, 57, 9, tzinfo=tzlocal())}]

In [35]:
!aws s3 ls --recursive {est_gpt2_8bit.model_data}


2026-04-26 05:57:25    3425806 reutersgpt-train-2026-04-26-05-01-11-367/output/model.tar.gz


In [36]:
!aws s3 cp {est_gpt2_8bit.model_data} ./temp/gpt2-lora-8-bit-quant.tar.gz

download: s3://sagemaker-us-east-1-191013407134/reutersgpt-train-2026-04-26-05-01-11-367/output/model.tar.gz to temp/gpt2-lora-8-bit-quant.tar.gz


In [37]:
%cd temp
!ls -ltrh

/home/ec2-user/SageMaker/CURRENTFOCUS/LLMS_transformers_WIP/LLMfromScratch/NLP_tasks/FineTuneDecoder/temp
total 8.2M
-rw-rw-r-- 1 ec2-user ec2-user 756K Feb 28  2025 Quant_ReuterSDSgpt2fine-tuning (2).ipynb
-rw-rw-r-- 1 ec2-user ec2-user 840K Feb 28  2025 Quant_ReuterSDSgpt2fine-tuning (1).ipynb
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  1  2025 ckpt_Epoch_4_Prplxty_1.0384616768394737
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  1  2025 ckpt_Epoch_4_Prplxty_1.0191842654468437
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  1  2025 ckpt_Epoch_4_Prplxty_1.00977148872375
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  1  2025 ckpt_Epoch_2_Prplxty_9.256776574458915
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  2  2025 ckpt_Epoch_2_Prplxty_8.363786833972586
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  2  2025 ckpt_Epoch_2_Prplxty_8.29573556990836
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  4  2025 ckpt_Epoch_20_Prplxty_4.906924388591055
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  4  2025 ckpt_Epoch_30_Prplxty_3.6311811785448

In [38]:
#!gunzip -dc model.tar.gz |tar xvf -
#!ls -ltrh

###### We have some observations:
- Training run time was lower than even with 4bit fine tuning --> can be attributed to gradient accumulation -8
- -Differential model is very small (less than 4 MB)
- further there is a scope for increasing gradient accumulation steps. we will explore that in peft only training

In [39]:
%cd ..
%pwd

/home/ec2-user/SageMaker/CURRENTFOCUS/LLMS_transformers_WIP/LLMfromScratch/NLP_tasks/FineTuneDecoder


'/home/ec2-user/SageMaker/CURRENTFOCUS/LLMS_transformers_WIP/LLMfromScratch/NLP_tasks/FineTuneDecoder'

In [40]:
hparams = {
    'bs': 16,
    'lrate': 0.0008,
    'num_epochs': 4,
    'environ': 'aws',
    'block_size': 256,
    'ckpt': 'gpt2',
    'quant_bits': 0,
    'peft': 'lora',
    'grad_accum_steps': 16
    }
hparams

{'bs': 16,
 'lrate': 0.0008,
 'num_epochs': 4,
 'environ': 'aws',
 'block_size': 256,
 'ckpt': 'gpt2',
 'quant_bits': 0,
 'peft': 'lora',
 'grad_accum_steps': 16}

In [41]:
est_gpt2_loraonly = Estimator(
    image_uri=training_image_uri,
    role=role,
    source_dir='./scripts',
    entry_point='train-gpt2-reuters.py',
    instance_count=ic,
    instance_type=i_type,
    base_job_name='reutersgpt-train',
    hyperparameters=hparams,
    metric_definitions=metric_definitions,
    objective_type=objective_type,
    objective_metric_name=objective_metric_name
)

est_gpt2_loraonly.fit(
    inputs=data_channels,
    wait=True
)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: reutersgpt-train-2026-04-26-06-05-23-370


2026-04-26 06:05:27 Starting - Starting the training job...
2026-04-26 06:05:42 Starting - Preparing the instances for training...
2026-04-26 06:06:06 Downloading - Downloading input data...
2026-04-26 06:06:37 Downloading - Downloading the training image..................
2026-04-26 06:09:54 Training - Training image download completed. Training in progress..bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
2026-04-26 06:10:01,602 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2026-04-26 06:10:01,624 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-04-26 06:10:01,634 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succeed.
2026-04-26 06:10:01,642 sagemaker_pytorch_container.training INFO     Invoking user training script.
2026-04-26 06:10:03,465 sagemaker-training-toolkit INFO     Installing dependencies 

In [42]:
est_gpt2_loraonly.latest_training_job.describe()['FinalMetricDataList']

[{'MetricName': 'Perplexity',
  'Value': 12.134512901306152,
  'Timestamp': datetime.datetime(2026, 4, 26, 7, 8, 30, tzinfo=tzlocal())},
 {'MetricName': 'average training loss',
  'Value': 2.496053695678711,
  'Timestamp': datetime.datetime(2026, 4, 26, 7, 8, 30, tzinfo=tzlocal())}]

In [43]:
!aws s3 ls --recursive {est_gpt2_loraonly.model_data}


2026-04-26 07:08:47    3426530 reutersgpt-train-2026-04-26-06-05-23-370/output/model.tar.gz


In [44]:
!aws s3 cp {est_gpt2_loraonly.model_data} ./temp/gpt2-lora-noquant.tar.gz

download: s3://sagemaker-us-east-1-191013407134/reutersgpt-train-2026-04-26-06-05-23-370/output/model.tar.gz to temp/gpt2-lora-noquant.tar.gz


In [45]:
%cd temp
!ls -ltrh

/home/ec2-user/SageMaker/CURRENTFOCUS/LLMS_transformers_WIP/LLMfromScratch/NLP_tasks/FineTuneDecoder/temp
total 12M
-rw-rw-r-- 1 ec2-user ec2-user 756K Feb 28  2025 Quant_ReuterSDSgpt2fine-tuning (2).ipynb
-rw-rw-r-- 1 ec2-user ec2-user 840K Feb 28  2025 Quant_ReuterSDSgpt2fine-tuning (1).ipynb
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  1  2025 ckpt_Epoch_4_Prplxty_1.0384616768394737
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  1  2025 ckpt_Epoch_4_Prplxty_1.0191842654468437
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  1  2025 ckpt_Epoch_4_Prplxty_1.00977148872375
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  1  2025 ckpt_Epoch_2_Prplxty_9.256776574458915
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  2  2025 ckpt_Epoch_2_Prplxty_8.363786833972586
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  2  2025 ckpt_Epoch_2_Prplxty_8.29573556990836
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  4  2025 ckpt_Epoch_20_Prplxty_4.906924388591055
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  4  2025 ckpt_Epoch_30_Prplxty_3.63118117854489

In [46]:
#!gunzip -dc model.tar.gz |tar xvf -
#!ls -ltrh

###### We have some observations:
- Training run time was higher than previous runs --> can be attributed to non-quantization
- Differential model is very small (less than 4 MB) as in previous runs

next we move with gpt2-medium. we will run only quantized runs and prhaps peft only rnus

In [47]:
%cd ..
%pwd

/home/ec2-user/SageMaker/CURRENTFOCUS/LLMS_transformers_WIP/LLMfromScratch/NLP_tasks/FineTuneDecoder


'/home/ec2-user/SageMaker/CURRENTFOCUS/LLMS_transformers_WIP/LLMfromScratch/NLP_tasks/FineTuneDecoder'

In [49]:
hparams = {
    'bs': 16,
    'lrate': 0.0008,
    'num_epochs': 2,
    'environ': 'aws',
    'block_size': 256,
    'ckpt': 'gpt2-medium',
    'quant_bits': 4,
    'peft': 'lora',
    'grad_accum_steps': 8
}
hparams

{'bs': 16,
 'lrate': 0.0008,
 'num_epochs': 2,
 'environ': 'aws',
 'block_size': 256,
 'ckpt': 'gpt2-medium',
 'quant_bits': 4,
 'peft': 'lora',
 'grad_accum_steps': 8}

In [50]:
est_gpt2_medium_4bit = Estimator(
    image_uri=training_image_uri,
    role=role,
    source_dir='./scripts',
    entry_point='train-gpt2-reuters.py',
    instance_count=ic,
    instance_type=i_type,
    base_job_name='reutersgpt-train',
    hyperparameters=hparams,
    metric_definitions=metric_definitions,
    objective_type=objective_type,
    objective_metric_name=objective_metric_name
)

est_gpt2_medium_4bit.fit(
    inputs=data_channels,
    wait=True
)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: reutersgpt-train-2026-04-26-07-14-52-226


2026-04-26 07:14:54 Starting - Starting the training job...
2026-04-26 07:15:10 Starting - Preparing the instances for training...
2026-04-26 07:15:33 Downloading - Downloading input data...
2026-04-26 07:15:59 Downloading - Downloading the training image..................
2026-04-26 07:19:16 Training - Training image download completed. Training in progress..bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
2026-04-26 07:19:28,039 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2026-04-26 07:19:28,057 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-04-26 07:19:28,067 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succeed.
2026-04-26 07:19:28,074 sagemaker_pytorch_container.training INFO     Invoking user training script.
2026-04-26 07:19:29,945 sagemaker-training-toolkit INFO     Installing dependencies 

In [51]:
est_gpt2_medium_4bit.latest_training_job.describe()['FinalMetricDataList']

[{'MetricName': 'Perplexity',
  'Value': 9.246621131896973,
  'Timestamp': datetime.datetime(2026, 4, 26, 8, 29, 24, tzinfo=tzlocal())},
 {'MetricName': 'average training loss',
  'Value': 2.2242581844329834,
  'Timestamp': datetime.datetime(2026, 4, 26, 8, 29, 24, tzinfo=tzlocal())}]

In [52]:
!aws s3 ls --recursive {est_gpt2_medium_4bit.model_data}

2026-04-26 08:29:39    7077043 reutersgpt-train-2026-04-26-07-14-52-226/output/model.tar.gz


In [53]:
!aws s3 cp {est_gpt2_medium_4bit.model_data} ./temp/gpt2-medium-lora-4-bit-quant.tar.gz

download: s3://sagemaker-us-east-1-191013407134/reutersgpt-train-2026-04-26-07-14-52-226/output/model.tar.gz to temp/gpt2-medium-lora-4-bit-quant.tar.gz


In [54]:
%cd temp
!ls -ltrh

/home/ec2-user/SageMaker/CURRENTFOCUS/LLMS_transformers_WIP/LLMfromScratch/NLP_tasks/FineTuneDecoder/temp
total 19M
-rw-rw-r-- 1 ec2-user ec2-user 756K Feb 28  2025 Quant_ReuterSDSgpt2fine-tuning (2).ipynb
-rw-rw-r-- 1 ec2-user ec2-user 840K Feb 28  2025 Quant_ReuterSDSgpt2fine-tuning (1).ipynb
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  1  2025 ckpt_Epoch_4_Prplxty_1.0384616768394737
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  1  2025 ckpt_Epoch_4_Prplxty_1.0191842654468437
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  1  2025 ckpt_Epoch_4_Prplxty_1.00977148872375
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  1  2025 ckpt_Epoch_2_Prplxty_9.256776574458915
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  2  2025 ckpt_Epoch_2_Prplxty_8.363786833972586
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  2  2025 ckpt_Epoch_2_Prplxty_8.29573556990836
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  4  2025 ckpt_Epoch_20_Prplxty_4.906924388591055
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  4  2025 ckpt_Epoch_30_Prplxty_3.63118117854489

Differential model is very small (about 6 MB). It took 1 hour 12 minutes to complete 2 epochs

In [55]:
%cd ..
%pwd

/home/ec2-user/SageMaker/CURRENTFOCUS/LLMS_transformers_WIP/LLMfromScratch/NLP_tasks/FineTuneDecoder


'/home/ec2-user/SageMaker/CURRENTFOCUS/LLMS_transformers_WIP/LLMfromScratch/NLP_tasks/FineTuneDecoder'

In [58]:
hparams = {
    'bs': 8,
    'lrate': 0.0008,
    'num_epochs': 2,
    'environ': 'aws',
    'block_size': 256,
    'ckpt': 'gpt2-medium',
    'quant_bits': 8,
    'peft': "lora",
    'grad_accum_steps': 4 #set to 4 if cuda errors out
}
hparams

{'bs': 8,
 'lrate': 0.0008,
 'num_epochs': 2,
 'environ': 'aws',
 'block_size': 256,
 'ckpt': 'gpt2-medium',
 'quant_bits': 8,
 'peft': 'lora',
 'grad_accum_steps': 4}

In [59]:
est_gpt2_medium_8bit = Estimator(
    image_uri=training_image_uri,
    role=role,
    source_dir='./scripts',
    entry_point='train-gpt2-reuters.py',
    instance_count=ic,
    instance_type=i_type,
    base_job_name='reutersgpt-train',
    hyperparameters=hparams,
    metric_definitions=metric_definitions,
    objective_type=objective_type,
    objective_metric_name=objective_metric_name
)

est_gpt2_medium_8bit.fit(
    inputs=data_channels,
    wait=True
)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: reutersgpt-train-2026-04-26-08-36-23-555


2026-04-26 08:36:25 Starting - Starting the training job...
2026-04-26 08:36:40 Starting - Preparing the instances for training...
2026-04-26 08:37:03 Downloading - Downloading input data...
2026-04-26 08:37:33 Downloading - Downloading the training image..................
2026-04-26 08:40:51 Training - Training image download completed. Training in progress..bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
2026-04-26 08:41:02,253 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2026-04-26 08:41:02,273 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-04-26 08:41:02,283 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succeed.
2026-04-26 08:41:02,290 sagemaker_pytorch_container.training INFO     Invoking user training script.
2026-04-26 08:41:04,256 sagemaker-training-toolkit INFO     Installing dependencies 

In [60]:
est_gpt2_medium_8bit.latest_training_job.describe()['FinalMetricDataList']

[{'MetricName': 'Perplexity',
  'Value': 8.337007522583008,
  'Timestamp': datetime.datetime(2026, 4, 26, 9, 41, 48, tzinfo=tzlocal())},
 {'MetricName': 'average training loss',
  'Value': 2.120704412460327,
  'Timestamp': datetime.datetime(2026, 4, 26, 9, 41, 48, tzinfo=tzlocal())}]

In [61]:
!aws s3 ls --recursive {est_gpt2_medium_8bit.model_data}
!aws s3 cp {est_gpt2_medium_8bit.model_data} ./temp/gpt2-medium-lora-8-bit-quant.tar.gz
%cd temp
!ls -ltrh

2026-04-26 09:42:08    7078966 reutersgpt-train-2026-04-26-08-36-23-555/output/model.tar.gz
download: s3://sagemaker-us-east-1-191013407134/reutersgpt-train-2026-04-26-08-36-23-555/output/model.tar.gz to temp/gpt2-medium-lora-8-bit-quant.tar.gz
/home/ec2-user/SageMaker/CURRENTFOCUS/LLMS_transformers_WIP/LLMfromScratch/NLP_tasks/FineTuneDecoder/temp
total 25M
-rw-rw-r-- 1 ec2-user ec2-user 756K Feb 28  2025 Quant_ReuterSDSgpt2fine-tuning (2).ipynb
-rw-rw-r-- 1 ec2-user ec2-user 840K Feb 28  2025 Quant_ReuterSDSgpt2fine-tuning (1).ipynb
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  1  2025 ckpt_Epoch_4_Prplxty_1.0384616768394737
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  1  2025 ckpt_Epoch_4_Prplxty_1.0191842654468437
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  1  2025 ckpt_Epoch_4_Prplxty_1.00977148872375
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  1  2025 ckpt_Epoch_2_Prplxty_9.256776574458915
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  2  2025 ckpt_Epoch_2_Prplxty_8.363786833972586
drwxr-xr-x 2 ec2-u

In [62]:
%cd ..
%pwd

/home/ec2-user/SageMaker/CURRENTFOCUS/LLMS_transformers_WIP/LLMfromScratch/NLP_tasks/FineTuneDecoder


'/home/ec2-user/SageMaker/CURRENTFOCUS/LLMS_transformers_WIP/LLMfromScratch/NLP_tasks/FineTuneDecoder'

In [64]:
hparams = {
    'bs': 8,
    'lrate': 0.0008,
    'num_epochs': 2,
    'environ': 'aws',
    'block_size': 256,
    'ckpt': 'gpt2-medium',
    'quant_bits': 0,
    'peft': "lora",
    'grad_accum_steps': 4 #set to 4 if cuda errors out
}
hparams

{'bs': 8,
 'lrate': 0.0008,
 'num_epochs': 2,
 'environ': 'aws',
 'block_size': 256,
 'ckpt': 'gpt2-medium',
 'quant_bits': 0,
 'peft': 'lora',
 'grad_accum_steps': 4}

In [65]:
est_gpt2_medium_peft = Estimator(
    image_uri=training_image_uri,
    role=role,
    source_dir='./scripts',
    entry_point='train-gpt2-reuters.py',
    instance_count=ic,
    instance_type=i_type,
    base_job_name='reutersgpt-train',
    hyperparameters=hparams,
    metric_definitions=metric_definitions,
    objective_type=objective_type,
    objective_metric_name=objective_metric_name
)

est_gpt2_medium_peft.fit(
    inputs=data_channels,
    wait=True
)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: reutersgpt-train-2026-04-26-09-45-38-009


2026-04-26 09:45:40 Starting - Starting the training job...
2026-04-26 09:45:55 Starting - Preparing the instances for training...
2026-04-26 09:46:20 Downloading - Downloading input data...
2026-04-26 09:46:45 Downloading - Downloading the training image..................
2026-04-26 09:50:07 Training - Training image download completed. Training in progress...bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
2026-04-26 09:50:18,825 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2026-04-26 09:50:18,844 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-04-26 09:50:18,855 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succeed.
2026-04-26 09:50:18,861 sagemaker_pytorch_container.training INFO     Invoking user training script.
2026-04-26 09:50:20,766 sagemaker-training-toolkit INFO     Installing dependencies

In [66]:
est_gpt2_medium_peft.latest_training_job.describe()['FinalMetricDataList']

[{'MetricName': 'Perplexity',
  'Value': 8.287935256958008,
  'Timestamp': datetime.datetime(2026, 4, 26, 10, 58, 52, tzinfo=tzlocal())},
 {'MetricName': 'average training loss',
  'Value': 2.1148009300231934,
  'Timestamp': datetime.datetime(2026, 4, 26, 10, 58, 52, tzinfo=tzlocal())}]

In [67]:
!aws s3 ls --recursive {est_gpt2_medium_peft.model_data}
!aws s3 cp {est_gpt2_medium_peft.model_data} ./temp/gpt2-medium-lora-noquant.tar
%cd temp
!ls -ltrh

2026-04-26 10:59:10    7079477 reutersgpt-train-2026-04-26-09-45-38-009/output/model.tar.gz
download: s3://sagemaker-us-east-1-191013407134/reutersgpt-train-2026-04-26-09-45-38-009/output/model.tar.gz to temp/gpt2-medium-lora-noquant.tar
/home/ec2-user/SageMaker/CURRENTFOCUS/LLMS_transformers_WIP/LLMfromScratch/NLP_tasks/FineTuneDecoder/temp
total 32M
-rw-rw-r-- 1 ec2-user ec2-user 756K Feb 28  2025 Quant_ReuterSDSgpt2fine-tuning (2).ipynb
-rw-rw-r-- 1 ec2-user ec2-user 840K Feb 28  2025 Quant_ReuterSDSgpt2fine-tuning (1).ipynb
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  1  2025 ckpt_Epoch_4_Prplxty_1.0384616768394737
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  1  2025 ckpt_Epoch_4_Prplxty_1.0191842654468437
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  1  2025 ckpt_Epoch_4_Prplxty_1.00977148872375
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  1  2025 ckpt_Epoch_2_Prplxty_9.256776574458915
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  2  2025 ckpt_Epoch_2_Prplxty_8.363786833972586
drwxr-xr-x 2 ec2-user ec2

In [68]:
%cd ..
%pwd

/home/ec2-user/SageMaker/CURRENTFOCUS/LLMS_transformers_WIP/LLMfromScratch/NLP_tasks/FineTuneDecoder


'/home/ec2-user/SageMaker/CURRENTFOCUS/LLMS_transformers_WIP/LLMfromScratch/NLP_tasks/FineTuneDecoder'

In [69]:
hparams = {
    'bs': 8,
    'lrate': 0.0008,
    'num_epochs': 2,
    'environ': 'aws',
    'block_size': 256,
    'ckpt': 'gpt2',
    'quant_bits': 0,
    'peft': "nolora",
    'grad_accum_steps': 4 #set to 4 if cuda errors out
}
hparams

{'bs': 8,
 'lrate': 0.0008,
 'num_epochs': 2,
 'environ': 'aws',
 'block_size': 256,
 'ckpt': 'gpt2',
 'quant_bits': 0,
 'peft': 'nolora',
 'grad_accum_steps': 4}

In [70]:
est_gpt2_only = Estimator(
    image_uri=training_image_uri,
    role=role,
    source_dir='./scripts',
    entry_point='train-gpt2-reuters.py',
    instance_count=ic,
    instance_type=i_type,
    base_job_name='reutersgpt-train',
    hyperparameters=hparams,
    metric_definitions=metric_definitions,
    objective_type=objective_type,
    objective_metric_name=objective_metric_name
)

est_gpt2_only.fit(
    inputs=data_channels,
    wait=True
)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: reutersgpt-train-2026-04-26-11-12-51-336


2026-04-26 11:12:55 Starting - Starting the training job...
2026-04-26 11:13:11 Starting - Preparing the instances for training...
2026-04-26 11:13:35 Downloading - Downloading input data...
2026-04-26 11:14:00 Downloading - Downloading the training image..................
2026-04-26 11:17:17 Training - Training image download completed. Training in progress..bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
2026-04-26 11:17:28,854 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2026-04-26 11:17:28,874 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-04-26 11:17:28,884 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succeed.
2026-04-26 11:17:28,891 sagemaker_pytorch_container.training INFO     Invoking user training script.
2026-04-26 11:17:30,812 sagemaker-training-toolkit INFO     Installing dependencies 

In [72]:
est_gpt2_only.latest_training_job.describe()['FinalMetricDataList']

[{'MetricName': 'Perplexity',
  'Value': 6.104443550109863,
  'Timestamp': datetime.datetime(2026, 4, 26, 11, 53, 28, tzinfo=tzlocal())},
 {'MetricName': 'average training loss',
  'Value': 1.8090169429779053,
  'Timestamp': datetime.datetime(2026, 4, 26, 11, 53, 28, tzinfo=tzlocal())}]

In [74]:
est_gpt2_only.latest_training_job.describe()

{'TrainingJobName': 'reutersgpt-train-2026-04-26-11-12-51-336',
 'TrainingJobArn': 'arn:aws:sagemaker:us-east-1:191013407134:training-job/reutersgpt-train-2026-04-26-11-12-51-336',
 'ModelArtifacts': {'S3ModelArtifacts': 's3://sagemaker-us-east-1-191013407134/reutersgpt-train-2026-04-26-11-12-51-336/output/model.tar.gz'},
 'TrainingJobStatus': 'Completed',
 'SecondaryStatus': 'Completed',
 'HyperParameters': {'block_size': '256',
  'bs': '8',
  'ckpt': '"gpt2"',
  'environ': '"aws"',
  'grad_accum_steps': '4',
  'lrate': '0.0008',
  'num_epochs': '2',
  'peft': '"nolora"',
  'quant_bits': '0',
  'sagemaker_container_log_level': '20',
  'sagemaker_job_name': '"reutersgpt-train-2026-04-26-11-12-51-336"',
  'sagemaker_program': '"train-gpt2-reuters.py"',
  'sagemaker_region': '"us-east-1"',
  'sagemaker_submit_directory': '"s3://sagemaker-us-east-1-191013407134/reutersgpt-train-2026-04-26-11-12-51-336/source/sourcedir.tar.gz"'},
 'AlgorithmSpecification': {'TrainingImage': '763104351884.d

In [82]:
est_gpt2_only.latest_training_job.describe()['FinalMetricDataList'][0]['Value']

6.104443550109863

In [79]:
est_gpt2_only.latest_training_job.describe()['HyperParameters']

{'block_size': '256',
 'bs': '8',
 'ckpt': '"gpt2"',
 'environ': '"aws"',
 'grad_accum_steps': '4',
 'lrate': '0.0008',
 'num_epochs': '2',
 'peft': '"nolora"',
 'quant_bits': '0',
 'sagemaker_container_log_level': '20',
 'sagemaker_job_name': '"reutersgpt-train-2026-04-26-11-12-51-336"',
 'sagemaker_program': '"train-gpt2-reuters.py"',
 'sagemaker_region': '"us-east-1"',
 'sagemaker_submit_directory': '"s3://sagemaker-us-east-1-191013407134/reutersgpt-train-2026-04-26-11-12-51-336/source/sourcedir.tar.gz"'}

In [100]:
!aws s3 ls --recursive {est_gpt2_only.model_data}
!aws s3 cp {est_gpt2_only.model_data} ./temp/gpt2-nolora-noquant.tar


2026-04-26 11:54:07 1376083371 reutersgpt-train-2026-04-26-11-12-51-336/output/model.tar.gz
download: s3://sagemaker-us-east-1-191013407134/reutersgpt-train-2026-04-26-11-12-51-336/output/model.tar.gz to temp/gpt2-nolora-noquant.tar
total 1.4G
-rw-rw-r-- 1 ec2-user ec2-user 756K Feb 28  2025 Quant_ReuterSDSgpt2fine-tuning (2).ipynb
-rw-rw-r-- 1 ec2-user ec2-user 840K Feb 28  2025 Quant_ReuterSDSgpt2fine-tuning (1).ipynb
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  1  2025 ckpt_Epoch_4_Prplxty_1.0384616768394737
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  1  2025 ckpt_Epoch_4_Prplxty_1.0191842654468437
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  1  2025 ckpt_Epoch_4_Prplxty_1.00977148872375
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  1  2025 ckpt_Epoch_2_Prplxty_9.256776574458915
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  2  2025 ckpt_Epoch_2_Prplxty_8.363786833972586
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  2  2025 ckpt_Epoch_2_Prplxty_8.29573556990836
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  4  2025

In [103]:
!ls -ltrh ./temp|grep 'gpt2-'

-rw-rw-r-- 1 ec2-user ec2-user 3.3M Apr 26 04:51 gpt2-lora-4-bit-quant.tar.gz
-rw-rw-r-- 1 ec2-user ec2-user 3.3M Apr 26 05:57 gpt2-lora-8-bit-quant.tar.gz
-rw-rw-r-- 1 ec2-user ec2-user 3.3M Apr 26 07:08 gpt2-lora-noquant.tar.gz
-rw-rw-r-- 1 ec2-user ec2-user 6.8M Apr 26 08:29 gpt2-medium-lora-4-bit-quant.tar.gz
-rw-rw-r-- 1 ec2-user ec2-user 6.8M Apr 26 09:42 gpt2-medium-lora-8-bit-quant.tar.gz
-rw-rw-r-- 1 ec2-user ec2-user 6.8M Apr 26 10:59 gpt2-medium-lora-noquant.tar.gz
-rw-rw-r-- 1 ec2-user ec2-user 1.3G Apr 26 11:54 gpt2-nolora-noquant.tar.gz


In [71]:
#Collect loss and perplexity for all 7 experiments
#create a pandas dataframe with following fields: Experiment name, loss value truncated to 3 places, perplexity
#truncated to 3 places

#next create a function that will load peft model with main model and then perform sample generation.
#Measure Bleu score and put it in a pandas dataframe
#DO this for all 7 experiments

In [78]:
!aws sagemaker list-training-jobs --creation-time-after 2026-04-25

{
    "TrainingJobSummaries": [
        {
            "TrainingJobName": "reutersgpt-train-2026-04-26-11-12-51-336",
            "TrainingJobArn": "arn:aws:sagemaker:us-east-1:191013407134:training-job/reutersgpt-train-2026-04-26-11-12-51-336",
            "CreationTime": 1777201972.449,
            "TrainingEndTime": 1777204457.107,
            "LastModifiedTime": 1777204457.236,
            "TrainingJobStatus": "Completed",
            "SecondaryStatus": "Completed"
        },
        {
            "TrainingJobName": "reutersgpt-train-2026-04-26-09-45-38-009",
            "TrainingJobArn": "arn:aws:sagemaker:us-east-1:191013407134:training-job/reutersgpt-train-2026-04-26-09-45-38-009",
            "CreationTime": 1777196738.583,
            "TrainingEndTime": 1777201155.355,
            "LastModifiedTime": 1777201155.53,
            "TrainingJobStatus": "Completed",
            "SecondaryStatus": "Completed"
        },
        {
            "TrainingJobName": "reutersgpt-train-2026-0

In [84]:
joblist = [est, est_gpt2_8bit, est_gpt2_loraonly, est_gpt2_medium_4bit, 
           est_gpt2_medium_8bit, est_gpt2_medium_peft, est_gpt2_only]

In [85]:
joblist

In [92]:
exp_hp = []
exp_loss = []
exp_perp = []
for job in joblist:
    exp_hp.append(job.latest_training_job.describe()['HyperParameters'])
    exp_perp.append(round(job.latest_training_job.describe()['FinalMetricDataList'][0]['Value'], 3))
    exp_loss.append(round(job.latest_training_job.describe()['FinalMetricDataList'][1]['Value'], 3))

In [93]:
exp_perp

[10.314, 11.326, 12.135, 9.247, 8.337, 8.288, 6.104]

In [96]:
exp_hp[0]

{'block_size': '256',
 'bs': '16',
 'ckpt': '"gpt2"',
 'environ': '"aws"',
 'lrate': '0.0006',
 'num_epochs': '4',
 'peft': '"lora"',
 'quant_bits': '4',
 'sagemaker_container_log_level': '20',
 'sagemaker_job_name': '"reutersgpt-train-2026-04-26-03-52-17-073"',
 'sagemaker_program': '"train-gpt2-reuters.py"',
 'sagemaker_region': '"us-east-1"',
 'sagemaker_submit_directory': '"s3://sagemaker-us-east-1-191013407134/reutersgpt-train-2026-04-26-03-52-17-073/source/sourcedir.tar.gz"'}

In [94]:
df = pd.DataFrame({'exp_hparams': exp_hp, 'loss': exp_loss, 'perplexity': exp_perp})
df

,exp_hparams,loss,perplexity
0,"{'block_size': '256', 'bs': '16', 'ckpt': '""gp...",2.334,10.314
1,"{'block_size': '256', 'bs': '16', 'ckpt': '""gp...",2.427,11.326
2,"{'block_size': '256', 'bs': '16', 'ckpt': '""gp...",2.496,12.135
3,"{'block_size': '256', 'bs': '16', 'ckpt': '""gp...",2.224,9.247
4,"{'block_size': '256', 'bs': '8', 'ckpt': '""gpt...",2.121,8.337
5,"{'block_size': '256', 'bs': '8', 'ckpt': '""gpt...",2.115,8.288
6,"{'block_size': '256', 'bs': '8', 'ckpt': '""gpt...",1.809,6.104


In [106]:
df['ckpt'] = df['exp_hparams'].apply(lambda x: x['ckpt'])
df['peft'] = df['exp_hparams'].apply(lambda x: x['peft'])
df['quant_bits'] = df['exp_hparams'].apply(lambda x: x['quant_bits'])
df

,exp_hparams,loss,perplexity,ckpt,peft,quant_bits
0,"{'block_size': '256', 'bs': '16', 'ckpt': '""gp...",2.334,10.314,"""gpt2""","""lora""",4
1,"{'block_size': '256', 'bs': '16', 'ckpt': '""gp...",2.427,11.326,"""gpt2""","""lora""",8
2,"{'block_size': '256', 'bs': '16', 'ckpt': '""gp...",2.496,12.135,"""gpt2""","""lora""",0
3,"{'block_size': '256', 'bs': '16', 'ckpt': '""gp...",2.224,9.247,"""gpt2-medium""","""lora""",4
4,"{'block_size': '256', 'bs': '8', 'ckpt': '""gpt...",2.121,8.337,"""gpt2-medium""","""lora""",8
5,"{'block_size': '256', 'bs': '8', 'ckpt': '""gpt...",2.115,8.288,"""gpt2-medium""","""lora""",0
6,"{'block_size': '256', 'bs': '8', 'ckpt': '""gpt...",1.809,6.104,"""gpt2""","""nolora""",0


In [14]:
ds_1['train']['full_article'][2]

"TITLE: TEXAS COMMERCE BANCSHARES <TCB> FILES PLAN\n\nBODY: Texas Commerce Bancshares Inc's Texas\nCommerce Bank-Houston said it filed an application with the\nComptroller of the Currency in an effort to create the largest\nbanking network in Harris County.\n    The bank said the network would link 31 banks having\n13.5 billion dlrs in assets and 7.5 billion dlrs in deposits.\n       \n Reuter\n\x03"

In [113]:
!ls -ltrh ./temp |grep 'gpt2-'

-rw-rw-r-- 1 ec2-user ec2-user 3.3M Apr 26 04:51 gpt2-lora-4-bit-quant.tar.gz
-rw-rw-r-- 1 ec2-user ec2-user 3.3M Apr 26 05:57 gpt2-lora-8-bit-quant.tar.gz
-rw-rw-r-- 1 ec2-user ec2-user 3.3M Apr 26 07:08 gpt2-lora-noquant.tar.gz
-rw-rw-r-- 1 ec2-user ec2-user 6.8M Apr 26 08:29 gpt2-medium-lora-4-bit-quant.tar.gz
-rw-rw-r-- 1 ec2-user ec2-user 6.8M Apr 26 09:42 gpt2-medium-lora-8-bit-quant.tar.gz
-rw-rw-r-- 1 ec2-user ec2-user 6.8M Apr 26 10:59 gpt2-medium-lora-noquant.tar.gz
-rw-rw-r-- 1 ec2-user ec2-user 1.3G Apr 26 11:54 gpt2-nolora-noquant.tar.gz


In [45]:
def load_model_gen_text(ckpt, lora_ckpt, prompt, num_toks, lora=True):
    model = AutoModelForCausalLM.from_pretrained(ckpt)
    if lora:
        model = PeftModel.from_pretrained(model, lora_ckpt)
    else:
        with open(lora_ckpt, 'rb') as f:
            checkpoint = torch.load(f, map_location=torch.device(device))
            print(f'checkpoint keys: {checkpoint.keys()}')
            model.load_state_dict(checkpoint['model_state_dict'])
            print("completed success")
    tokenizer = AutoTokenizer.from_pretrained(ckpt)
    input_ids = tokenizer.encode(prompt, return_tensors='pt')
    gen_tokens = model.generate(
        input_ids,
        do_sample=True,
        temperature=0.8,
        max_length=num_toks,
    )
    gen_text = tokenizer.batch_decode(gen_tokens)[0]
    print(f'Prompt: {prompt}')
    print(gen_text)
    del model

In [20]:
#Collect loss and perplexity for all 7 experiments
#create a pandas dataframe with following fields: Experiment name, loss value truncated to 3 places, perplexity
#truncated to 3 places

#next create a function that will load peft model with main model and then perform sample generation.
#Measure Bleu score and put it in a pandas dataframe
#DO this for all 7 experiments

In [21]:
prompt = "TITLE: TEXAS COMMERCE BANCSHARES <TCB> FILES PLAN\n\nBODY:"
prompt

'TITLE: TEXAS COMMERCE BANCSHARES <TCB> FILES PLAN\n\nBODY:'

In [22]:
num_toks=512
num_toks

512

In [23]:
!ls -ltrh ./temp |grep 'gpt2-'

-rw-rw-r-- 1 ec2-user ec2-user 3.3M Apr 26 04:51 gpt2-lora-4-bit-quant.tar.gz
-rw-rw-r-- 1 ec2-user ec2-user 3.3M Apr 26 05:57 gpt2-lora-8-bit-quant.tar.gz
-rw-rw-r-- 1 ec2-user ec2-user 3.3M Apr 26 07:08 gpt2-lora-noquant.tar.gz
-rw-rw-r-- 1 ec2-user ec2-user 6.8M Apr 26 08:29 gpt2-medium-lora-4-bit-quant.tar.gz
-rw-rw-r-- 1 ec2-user ec2-user 6.8M Apr 26 09:42 gpt2-medium-lora-8-bit-quant.tar.gz
-rw-rw-r-- 1 ec2-user ec2-user 6.8M Apr 26 10:59 gpt2-medium-lora-noquant.tar.gz
-rw-rw-r-- 1 ec2-user ec2-user 1.3G Apr 26 11:54 gpt2-nolora-noquant.tar.gz


In [13]:
!gunzip -dc ./temp/gpt2-lora-4-bit-quant.tar.gz |tar xvf -

tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_4_Prplxty_10.314107140187565/
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_4_Prplxty_10.314107140187565/training_args.bin
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_4_Prplxty_10.314107140187565/tokenizer.json
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_4_Prplxty_10.314107140187565/README.md
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_4_Prplxty_10.314107140187565/merges.txt
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_4_Prplxty_10.314107140187565/adapter_model.safetensors
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_4_Prplxty_10.314107140187565/vocab.json
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_4_Prplxty_10.314107140187565/adapter_conf

In [26]:
load_model_gen_text('gpt2', 'ckpt_Epoch_4_Prplxty_10.314107140187565', prompt, num_toks)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Prompt: TITLE: TEXAS COMMERCE BANCSHARES <TCB> FILES PLAN

BODY:
TITLE: TEXAS COMMERCE BANCSHARES <TCB> FILES PLAN

BODY: Texas Commuter Bank Holdings Inc said
it filed with the Securities and Exchange Commission a
plan to make deposits and accept cash. The plan will be
filed as part of its initial public offering, the company said.
    It said the Securities and Exchange Commission will begin
to act as a regulatory agency rather than a trading
person.
 Reuter
<|endoftext|>


In [27]:
!gunzip -dc ./temp/gpt2-lora-8-bit-quant.tar.gz |tar xvf -

tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_4_Prplxty_11.326158550842958/
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_4_Prplxty_11.326158550842958/vocab.json
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_4_Prplxty_11.326158550842958/merges.txt
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_4_Prplxty_11.326158550842958/adapter_model.safetensors
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_4_Prplxty_11.326158550842958/README.md
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_4_Prplxty_11.326158550842958/adapter_config.json
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_4_Prplxty_11.326158550842958/training_args.bin
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_4_Prplxty_11.326158550842958/special

In [28]:
load_model_gen_text('gpt2', 'ckpt_Epoch_4_Prplxty_11.326158550842958', prompt, num_toks)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt: TITLE: TEXAS COMMERCE BANCSHARES <TCB> FILES PLAN

BODY:
TITLE: TEXAS COMMERCE BANCSHARES <TCB> FILES PLAN

BODY: Texas Commerce and Construction Corp
said it filed a proposal to acquire all of Chevron Corp as
a result of an agreement reached in April 2016 between
the three companies.
    The Texas proposal has the approval of the U.S.
Committee on Foreign Investment in the United States.
    The Texas proposal is intended to improve the
construction of the Corpus Christi gas basin and to diversify the
resource resources of the state.            
 Reuter
quickShip<|endoftext|>


In [29]:
!gunzip -dc ./temp/gpt2-lora-noquant.tar.gz |tar xvf -

tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_4_Prplxty_12.134512543935335/
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_4_Prplxty_12.134512543935335/tokenizer_config.json
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_4_Prplxty_12.134512543935335/README.md
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_4_Prplxty_12.134512543935335/tokenizer.json
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_4_Prplxty_12.134512543935335/merges.txt
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_4_Prplxty_12.134512543935335/adapter_model.safetensors
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_4_Prplxty_12.134512543935335/special_tokens_map.json
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_4_Prplxty_12.13451254393

In [30]:
load_model_gen_text('gpt2', 'ckpt_Epoch_4_Prplxty_12.134512543935335', prompt, num_toks)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt: TITLE: TEXAS COMMERCE BANCSHARES <TCB> FILES PLAN

BODY:
TITLE: TEXAS COMMERCE BANCSHARES <TCB> FILES PLAN

BODY: The Texas Commerce Commission
approved two securities contracts of the Pacific Southwest Inc
<PWR> and the C.I.P. of Los Angeles Corp <LAD> to cover
$5.2 billion in capital expenses and to raise $1.2 billion
in financing, the commission reported.
    The securities awards and commissions are authorized through
a combination of two third party securities market
agents.
    The commission also approved a memorandum of understanding with the
Pacific Southwest to develop the $5.2 billion securities
contract with C.I.P. of Los Angeles, and C.I.P. of
Los Angeles, to secure a 1.25 billion dlr increase in
capital expenses.
    The commission also approved a 5.8 billion dlr, five-year
comprehensive, three-year capital agreement with the C.I.P.
of Los Angeles, to cover $25.5 billion in capital
increases and $1.3 billion in debt.
    The commission reported the deal<|endoftext

In [31]:
!gunzip -dc ./temp/gpt2-medium-lora-4-bit-quant.tar.gz |tar xvf -

tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_2_Prplxty_9.246621550015862/
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_2_Prplxty_9.246621550015862/adapter_config.json
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_2_Prplxty_9.246621550015862/tokenizer.json
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_2_Prplxty_9.246621550015862/merges.txt
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_2_Prplxty_9.246621550015862/training_args.bin
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_2_Prplxty_9.246621550015862/adapter_model.safetensors
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_2_Prplxty_9.246621550015862/special_tokens_map.json
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_2_Prplxty_9.2466215500158

In [32]:
load_model_gen_text('gpt2-medium', 'ckpt_Epoch_2_Prplxty_9.246621550015862', prompt, num_toks)

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt: TITLE: TEXAS COMMERCE BANCSHARES <TCB> FILES PLAN

BODY:
TITLE: TEXAS COMMERCE BANCSHARES <TCB> FILES PLAN

BODY: Texas Commerce Bancshares Inc said
its board has unanimously approved a plan to sell its
business to American Continental Corp <AFS> for an undisclosed
amount of money.
    The company said the transaction would be completed
by the end of the third quarter. It said the sale would
result in a net benefit to the Dallas-based company of
1,000,000 dlrs over five years.
    It said this is a move to address the company's
irrelevance to the Dallas market.
    The company said the board has approved a plan to
sell its stake in American Continental to American
Bancshares Inc, a Delaware-based company that is
associated with Continental and with the Texas-based
Texas-Dominion Bank.
    American Continental has about one mln dlrs of net
profit from the Texas commerce unit in the current year,
its chief executive, Roger Hall, said in a statement.
    This year, Texas Commerce 

In [33]:
!gunzip -dc ./temp/gpt2-medium-lora-8-bit-quant.tar.gz |tar xvf -

tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_2_Prplxty_8.337007894999958/
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_2_Prplxty_8.337007894999958/special_tokens_map.json
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_2_Prplxty_8.337007894999958/tokenizer.json
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_2_Prplxty_8.337007894999958/merges.txt
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_2_Prplxty_8.337007894999958/adapter_config.json
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_2_Prplxty_8.337007894999958/vocab.json
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_2_Prplxty_8.337007894999958/training_args.bin
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_2_Prplxty_8.337007894999958/tokenizer_co

In [34]:
load_model_gen_text('gpt2-medium', 'ckpt_Epoch_2_Prplxty_8.337007894999958', prompt, num_toks)

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt: TITLE: TEXAS COMMERCE BANCSHARES <TCB> FILES PLAN

BODY:
TITLE: TEXAS COMMERCE BANCSHARES <TCB> FILES PLAN

BODY: Texas Commerce Bancshares Inc
said it has filed with the Federal Trade Commission a
plan by the company to buy all the Texas Bancshares in
Texas and consolidate the companies.
    The company said in a filing that it has a plan to
consolidate the Texas Bancshares into a single company,
Texas Bancshares Texas Corp, with the purpose of selling any
Bancshares that may be acquired.
    The filing said Texas Commerce Bancshares plans to
consolidate the Bancshares into an exchange company, so
the company will not be subject to federal market
control laws.
 Reuter
<|endoftext|>


In [35]:
!gunzip -dc ./temp/gpt2-medium-lora-noquant.tar.gz |tar xvf -

tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_2_Prplxty_8.287935356408438/
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_2_Prplxty_8.287935356408438/vocab.json
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_2_Prplxty_8.287935356408438/tokenizer_config.json
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_2_Prplxty_8.287935356408438/special_tokens_map.json
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_2_Prplxty_8.287935356408438/tokenizer.json
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_2_Prplxty_8.287935356408438/training_args.bin
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_2_Prplxty_8.287935356408438/adapter_model.safetensors
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_2_Prplxty_8.28793535640

In [36]:
load_model_gen_text('gpt2-medium', 'ckpt_Epoch_2_Prplxty_8.287935356408438', prompt, num_toks)

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt: TITLE: TEXAS COMMERCE BANCSHARES <TCB> FILES PLAN

BODY:
TITLE: TEXAS COMMERCE BANCSHARES <TCB> FILES PLAN

BODY: Texas Commerce Bancshares Inc
said it filed for a Class B common stock tender offer to raise
350 mln dlrs through a series one subordinated debt offering.
    The company said it will sell Class B common stock at 30
dlrs per share, subject to adjustment for a reduction in
gross senior indebtedness.
 Reuter
<|endoftext|>


In [37]:
!gunzip -dc ./temp/gpt2-nolora-noquant.tar.gz |tar xvf -

tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_2_Prplxty_6.104443553348461.pt


In [50]:
device='cuda'
load_model_gen_text('gpt2', 'ckpt_Epoch_2_Prplxty_6.104443553348461.pt',  prompt=prompt, num_toks=num_toks, lora=False)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

checkpoint keys: dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'loss', 'device'])
completed success


[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt: TITLE: TEXAS COMMERCE BANCSHARES <TCB> FILES PLAN

BODY:
TITLE: TEXAS COMMERCE BANCSHARES <TCB> FILES PLAN

BODY: Texas Commerce Bancshares Inc's Dallas
Mortgage and Trust Co said it filed a registration statement
with the Securities and Exchange Commission for the proposed offering of
the securities.
    The company said proceeds of the offering will be used to
buy 2.5 mln shares of its common stock, for working capital,
to repay short-term bank debt, for working capital and
for working on the company's industrial revenue financing.
    Texas Commerce said proceeds of the offering will be used
to reduce short-term bank borrowings, for working capital,
for general corporate purposes and for working
capital.
 Reuter
<|endoftext|>
